# Phase 2 — Silver Layer: Cleanse, Standardize, Join

Builds a denormalized Silver orders table at order-item grain — one row
per (order, product) line item — enriched with customer, product, and
order-level payment total. Cleansing: drop rows missing `order_id` or
`customer_id`, dedupe on `order_id`, standardize
`order_purchase_timestamp` into an `order_date` column.

In [ ]:
from pyspark.sql import functions as F

## Load Bronze tables

In [ ]:
orders_bronze = spark.table("bronze.orders")
customers_bronze = spark.table("bronze.customers")
order_items_bronze = spark.table("bronze.order_items")
products_bronze = spark.table("bronze.products")
payments_bronze = spark.table("bronze.order_payments")
category_translation_bronze = spark.table("bronze.product_category_translation")

## Clean orders

Drop rows missing the join keys, dedupe on `order_id`, and derive
`order_date` from `order_purchase_timestamp`.

In [ ]:
orders_clean = (
    orders_bronze
    .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
    .dropDuplicates(["order_id"])
    .withColumn("order_date", F.to_date("order_purchase_timestamp"))
    .select("order_id", "customer_id", "order_status", "order_date")
)

## Derive order_amount from payments

`order_amount` doesn't exist as a column in the raw data — it's the sum
of `payment_value` per `order_id` (an order can have multiple payment
installments/methods).

In [ ]:
order_amount = (
    payments_bronze
    .groupBy("order_id")
    .agg(F.sum("payment_value").alias("order_amount"))
)

## Resolve product category to its English name

Joins `product_category_name` against the translation table so Gold
aggregates can group by a readable category name.

In [ ]:
products_translated = (
    products_bronze
    .join(category_translation_bronze, on="product_category_name", how="left")
    .select("product_id", "product_category_name", "product_category_name_english")
)

## Select customer identity columns

`customer_id` is effectively one-per-order in this dataset;
`customer_unique_id` is the actual customer identity, used for LTV in Gold.

In [ ]:
customers_clean = customers_bronze.select("customer_id", "customer_unique_id")

## Join into the Silver base table

Grain is order-item level (one row per line item) so each item keeps its
own `price`/`freight_value` for revenue aggregation, while `order_amount`
(the order-level payment total) and `order_date` are repeated across an
order's items.

In [ ]:
silver_orders = (
    order_items_bronze
    .select("order_id", "product_id", "price", "freight_value")
    .join(orders_clean, on="order_id", how="inner")
    .join(customers_clean, on="customer_id", how="inner")
    .join(products_translated, on="product_id", how="left")
    .join(order_amount, on="order_id", how="inner")
)

## Write Silver Delta table

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

silver_orders.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("silver.orders")

print(f"silver.orders: {spark.table('silver.orders').count()} rows")